# GeoRules Interactive Explorer

Interactively create and visualize 3D geological models. Select a model type, adjust parameters with sliders, and generate models on demand.

Supported layers: **Lobe**, **Gaussian**, **Meandering Channel**, **Braided Channel**, **Delta**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import georules as gr

%matplotlib inline

## Model Builder

1. Choose a geology type from the dropdown
2. Adjust the parameters using sliders — the plot updates automatically

In [2]:
# ── Shared grid widgets ──────────────────────────────────────────────
style = {'description_width': '160px'}
layout = widgets.Layout(width='440px')

w_nx = widgets.IntSlider(value=80, min=20, max=200, step=10, description='nx (grid X)', style=style, layout=layout)
w_ny = widgets.IntSlider(value=80, min=20, max=200, step=10, description='ny (grid Y)', style=style, layout=layout)
w_nz = widgets.IntSlider(value=30, min=5, max=80, step=5, description='nz (grid Z)', style=style, layout=layout)
w_xlen = widgets.IntSlider(value=2000, min=200, max=5000, step=200, description='x_len (m)', style=style, layout=layout)
w_ylen = widgets.IntSlider(value=2000, min=200, max=5000, step=200, description='y_len (m)', style=style, layout=layout)
w_zlen = widgets.IntSlider(value=80, min=10, max=300, step=10, description='z_len (m)', style=style, layout=layout)
w_depth = widgets.IntSlider(value=5000, min=1000, max=8000, step=100, description='top_depth (m)', style=style, layout=layout)
w_dip = widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5, description='dip (deg)', style=style, layout=layout)

grid_box = widgets.VBox([
    widgets.HTML('<b>Grid Parameters</b>'),
    widgets.HBox([widgets.VBox([w_nx, w_ny, w_nz, w_depth]),
                  widgets.VBox([w_xlen, w_ylen, w_zlen, w_dip])])
])

# ── Lobe widgets ─────────────────────────────────────────────────────
l_poro = widgets.FloatSlider(value=0.20, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)
l_perm = widgets.FloatSlider(value=1.5, min=0.1, max=5.0, step=0.1, description='perm_ave (log10 mD)', style=style, layout=layout)
l_poro_std = widgets.FloatSlider(value=0.03, min=0.005, max=0.10, step=0.005, description='poro_std', style=style, layout=layout)
l_perm_std = widgets.FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05, description='perm_std', style=style, layout=layout)
l_ntg = widgets.FloatSlider(value=0.7, min=0.1, max=1.0, step=0.05, description='NTG', style=style, layout=layout)
l_dhmin = widgets.FloatSlider(value=4.0, min=1.0, max=10.0, step=0.5, description='dhmin (height min)', style=style, layout=layout)
l_dhmax = widgets.FloatSlider(value=4.0, min=1.0, max=15.0, step=0.5, description='dhmax (height max)', style=style, layout=layout)
l_rmin = widgets.IntSlider(value=15, min=5, max=50, step=1, description='rmin (lobe radius)', style=style, layout=layout)
l_rmax = widgets.IntSlider(value=25, min=10, max=60, step=1, description='rmax (lobe radius)', style=style, layout=layout)
l_asp = widgets.FloatSlider(value=1.5, min=1.0, max=5.0, step=0.1, description='aspect ratio', style=style, layout=layout)
l_theta = widgets.FloatSlider(value=0.0, min=-90.0, max=90.0, step=5.0, description='theta0 (orientation)', style=style, layout=layout)
l_m = widgets.IntSlider(value=100, min=10, max=500, step=10, description='m (compensation)', style=style, layout=layout)
l_upthin = widgets.Checkbox(value=False, description='upthinning', style=style, layout=layout)
l_bouma = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='bouma_factor', style=style, layout=layout)

lobe_box = widgets.VBox([
    widgets.HTML('<b>Lobe Parameters</b>'),
    widgets.HBox([widgets.VBox([l_poro, l_perm, l_poro_std, l_perm_std, l_ntg, l_bouma, l_theta]),
                  widgets.VBox([l_dhmin, l_dhmax, l_rmin, l_rmax, l_asp, l_m, l_upthin])])
])

# ── Gaussian widgets ─────────────────────────────────────────────────
g_poro = widgets.FloatSlider(value=0.15, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)
g_perm = widgets.FloatSlider(value=1.2, min=0.1, max=5.0, step=0.1, description='perm_ave (log10 mD)', style=style, layout=layout)
g_poro_std = widgets.FloatSlider(value=0.03, min=0.005, max=0.10, step=0.005, description='poro_std', style=style, layout=layout)
g_perm_std = widgets.FloatSlider(value=0.4, min=0.05, max=2.0, step=0.05, description='perm_std', style=style, layout=layout)
g_ntg = widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description='NTG', style=style, layout=layout)
g_fx = widgets.FloatSlider(value=2.5, min=0.5, max=10.0, step=0.5, description='facies_filt X', style=style, layout=layout)
g_fy = widgets.FloatSlider(value=5.0, min=0.5, max=15.0, step=0.5, description='facies_filt Y', style=style, layout=layout)
g_fz = widgets.FloatSlider(value=2.5, min=0.5, max=10.0, step=0.5, description='facies_filt Z', style=style, layout=layout)
g_sx = widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.5, description='sand_filt X', style=style, layout=layout)
g_sy = widgets.FloatSlider(value=2.5, min=0.5, max=8.0, step=0.5, description='sand_filt Y', style=style, layout=layout)
g_sz = widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.5, description='sand_filt Z', style=style, layout=layout)
g_nugget = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='nugget', style=style, layout=layout)
g_corr = widgets.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.05, description='poro_perm_corr', style=style, layout=layout)

gauss_box = widgets.VBox([
    widgets.HTML('<b>Gaussian Parameters</b>'),
    widgets.HBox([widgets.VBox([g_poro, g_perm, g_poro_std, g_perm_std, g_ntg, g_nugget, g_corr]),
                  widgets.VBox([g_fx, g_fy, g_fz, g_sx, g_sy, g_sz])])
])

# ── Meandering Channel widgets ───────────────────────────────────────
c_width = widgets.IntSlider(value=80, min=20, max=300, step=10, description='channel_width (m)', style=style, layout=layout)
c_nch = widgets.IntSlider(value=5, min=1, max=20, step=1, description='n_channels', style=style, layout=layout)
c_dwr = widgets.FloatSlider(value=0.4, min=0.05, max=1.0, step=0.05, description='depth_width_ratio', style=style, layout=layout)
c_meander = widgets.FloatSlider(value=1.2, min=0.0, max=3.0, step=0.1, description='meander_scale', style=style, layout=layout)
c_avulsion = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='avulsion_prob', style=style, layout=layout)
c_poro = widgets.FloatSlider(value=0.3, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)
c_friction = widgets.FloatSlider(value=0.0009, min=0.0001, max=0.01, step=0.0001, readout_format='.4f', description='friction_coeff', style=style, layout=layout)
c_ampl = widgets.FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0, description='amplitude', style=style, layout=layout)
c_slope = widgets.FloatSlider(value=0.008, min=0.0005, max=0.05, step=0.0005, readout_format='.4f', description='slope', style=style, layout=layout)
c_discharge = widgets.FloatSlider(value=0.9, min=0.1, max=5.0, step=0.1, description='discharge', style=style, layout=layout)
c_mig = widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.1, description='migration_ratio', style=style, layout=layout)
c_reflect = widgets.Checkbox(value=True, description='boundary_reflect', style=style, layout=layout)
c_aggr = widgets.Dropdown(options=['discrete', 'continuous'], value='discrete', description='aggradation_mode', style=style, layout=layout)
c_nlevel = widgets.IntSlider(value=6, min=2, max=20, step=1, description='nlevel', style=style, layout=layout)
c_reseed = widgets.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.05, description='level_reseed_prob', style=style, layout=layout)
c_jump = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description='level_jump_ratio', style=style, layout=layout)
c_avul_in = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='prob_avul_inside', style=style, layout=layout)
c_avul_out = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='prob_avul_outside', style=style, layout=layout)
c_azim = widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=5.0, description='azimuth (deg)', style=style, layout=layout)
c_abmud = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='abandoned_mud_prop', style=style, layout=layout)

channel_box = widgets.VBox([
    widgets.HTML('<b>Meandering Channel Parameters</b>'),
    widgets.HBox([
        widgets.VBox([c_width, c_nch, c_dwr, c_meander, c_avulsion, c_poro,
                      c_aggr, c_nlevel, c_reseed]),
        widgets.VBox([c_friction, c_ampl, c_slope, c_discharge, c_mig, c_reflect,
                      c_jump, c_avul_in, c_avul_out, c_azim, c_abmud])
    ])
])

# ── Braided Channel widgets ──────────────────────────────────────────
b_bpw = widgets.IntSlider(value=500, min=100, max=1500, step=50, description='braidplain_width', style=style, layout=layout)
b_nch = widgets.IntSlider(value=24, min=1, max=60, step=1, description='n_channels', style=style, layout=layout)
b_dwr = widgets.FloatSlider(value=0.1, min=0.02, max=0.5, step=0.01, description='depth_width_ratio', style=style, layout=layout)
b_meander = widgets.FloatSlider(value=0.9, min=0.0, max=3.0, step=0.1, description='meander_scale', style=style, layout=layout)
b_mig = widgets.FloatSlider(value=0.3, min=0.05, max=3.0, step=0.05, description='migration_ratio', style=style, layout=layout)
b_avul_in = widgets.FloatSlider(value=0.55, min=0.0, max=1.0, step=0.05, description='prob_avul_inside', style=style, layout=layout)
b_avul_out = widgets.FloatSlider(value=0.30, min=0.0, max=1.0, step=0.05, description='prob_avul_outside', style=style, layout=layout)
b_nlevel = widgets.IntSlider(value=0, min=0, max=40, step=1, description='nlevel (0=auto)', style=style, layout=layout)
b_reseed = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description='level_reseed_prob', style=style, layout=layout)
b_jump = widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description='level_jump_ratio', style=style, layout=layout)
b_slope = widgets.FloatSlider(value=0.008, min=0.0005, max=0.05, step=0.0005, readout_format='.4f', description='slope', style=style, layout=layout)
b_discharge = widgets.FloatSlider(value=0.9, min=0.1, max=5.0, step=0.1, description='discharge', style=style, layout=layout)
b_ampl = widgets.FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0, description='amplitude', style=style, layout=layout)
b_friction = widgets.FloatSlider(value=0.0009, min=0.0001, max=0.01, step=0.0001, readout_format='.4f', description='friction_coeff', style=style, layout=layout)
b_poro = widgets.FloatSlider(value=0.3, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)
b_azim = widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=5.0, description='azimuth (deg)', style=style, layout=layout)
b_abmud = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='abandoned_mud_prop', style=style, layout=layout)

braided_box = widgets.VBox([
    widgets.HTML('<b>Braided Channel Parameters</b>'),
    widgets.HBox([
        widgets.VBox([b_bpw, b_nch, b_dwr, b_meander, b_mig, b_avul_in, b_avul_out, b_poro]),
        widgets.VBox([b_nlevel, b_reseed, b_jump, b_slope, b_discharge, b_ampl, b_friction, b_azim, b_abmud])
    ])
])

# ── Delta widgets ────────────────────────────────────────────────────
d_feeder = widgets.FloatSlider(value=60.0, min=10.0, max=300.0, step=5.0, description='feeder_width (m)', style=style, layout=layout)
d_ngen = widgets.IntSlider(value=8, min=1, max=30, step=1, description='n_generations', style=style, layout=layout)
d_fan = widgets.FloatSlider(value=95.0, min=20.0, max=180.0, step=5.0, description='fan_angle_deg', style=style, layout=layout)
d_bif = widgets.IntSlider(value=4, min=1, max=6, step=1, description='bifurcation_depth', style=style, layout=layout)
d_kmin = widgets.IntSlider(value=3, min=2, max=6, step=1, description='children_min', style=style, layout=layout)
d_kmax = widgets.IntSlider(value=5, min=2, max=8, step=1, description='children_max', style=style, layout=layout)
d_asym = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description='fan_asymmetry', style=style, layout=layout)
d_azim = widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=5.0, description='azimuth (deg)', style=style, layout=layout)
d_apex = widgets.FloatSlider(value=0.25, min=0.05, max=0.8, step=0.05, description='apex_x_fraction', style=style, layout=layout)
d_prog = widgets.FloatSlider(value=0.0, min=0.0, max=0.6, step=0.05, description='progradation_frac', style=style, layout=layout)
d_trunk = widgets.FloatSlider(value=5.5, min=1.0, max=15.0, step=0.5, description='trunk_length_factor', style=style, layout=layout)
d_ltap = widgets.FloatSlider(value=0.88, min=0.5, max=1.0, step=0.02, description='length_taper', style=style, layout=layout)
d_wtap = widgets.FloatSlider(value=0.80, min=0.3, max=1.0, step=0.02, description='width_taper', style=style, layout=layout)
d_mwig = widgets.FloatSlider(value=0.04, min=0.0, max=0.2, step=0.01, description='meander_amp', style=style, layout=layout)
d_mbL = widgets.FloatSlider(value=2.2, min=0.5, max=6.0, step=0.1, description='mb_length_factor', style=style, layout=layout)
d_mbW = widgets.FloatSlider(value=1.6, min=0.5, max=5.0, step=0.1, description='mb_width_factor', style=style, layout=layout)
d_mbhw = widgets.FloatSlider(value=0.06, min=0.01, max=0.3, step=0.01, description='mb_hw_ratio', style=style, layout=layout)
d_mbdw = widgets.FloatSlider(value=0.08, min=0.01, max=0.3, step=0.01, description='mb_dw_ratio', style=style, layout=layout)
d_dwr = widgets.FloatSlider(value=0.25, min=0.05, max=1.0, step=0.05, description='dwratio', style=style, layout=layout)
d_poro = widgets.FloatSlider(value=0.25, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)

delta_box = widgets.VBox([
    widgets.HTML('<b>Delta Parameters</b>'),
    widgets.HBox([
        widgets.VBox([d_feeder, d_ngen, d_fan, d_bif, d_kmin, d_kmax, d_asym, d_azim,
                      d_apex, d_prog, d_dwr]),
        widgets.VBox([d_trunk, d_ltap, d_wtap, d_mwig,
                      d_mbL, d_mbW, d_mbhw, d_mbdw, d_poro])
    ])
])

# ── View options ─────────────────────────────────────────────────────
w_view = widgets.Dropdown(
    options=['Cube Slices', 'Z Slices', 'X Slices', 'Y Slices'],
    value='Cube Slices', description='View', style=style, layout=layout
)
w_prop = widgets.Dropdown(
    options=['Porosity', 'Permeability', 'Facies'],
    value='Porosity', description='Property', style=style, layout=layout
)
view_box = widgets.VBox([
    widgets.HTML('<b>Visualization</b>'),
    widgets.HBox([w_view, w_prop])
])

print('Widgets ready.')

Widgets ready.


In [3]:
# ── Model selector & dynamic panel ───────────────────────────────────
model_selector = widgets.Dropdown(
    options=['Lobe', 'Gaussian', 'Meandering Channel', 'Braided Channel', 'Delta'],
    value='Lobe', description='Model Type', style=style, layout=layout
)

param_area = widgets.Output()
output_area = widgets.Output()

param_panels = {
    'Lobe': lobe_box,
    'Gaussian': gauss_box,
    'Meandering Channel': channel_box,
    'Braided Channel': braided_box,
    'Delta': delta_box,
}

def update_params(*args):
    with param_area:
        clear_output(wait=True)
        display(param_panels[model_selector.value])

model_selector.observe(update_params, names='value')

# ── Auto-generate on any widget change ───────────────────────────────
def generate_model(*args):
    with output_area:
        clear_output(wait=True)
        plt.close('all')
        model_type = model_selector.value
        print(f'Generating {model_type} model...')

        try:
            grid_kw = dict(
                nx=w_nx.value, ny=w_ny.value, nz=w_nz.value,
                x_len=w_xlen.value, y_len=w_ylen.value, z_len=w_zlen.value,
                top_depth=w_depth.value, dip=w_dip.value,
            )

            if model_type == 'Lobe':
                layer = gr.LobeLayer(**grid_kw)
                layer.create_geology(
                    poro_ave=l_poro.value, perm_ave=l_perm.value,
                    poro_std=l_poro_std.value, perm_std=l_perm_std.value,
                    ntg=l_ntg.value,
                    dhmin=l_dhmin.value, dhmax=l_dhmax.value,
                    rmin=l_rmin.value, rmax=l_rmax.value,
                    asp=l_asp.value, theta0=l_theta.value, m=l_m.value,
                    upthinning=l_upthin.value,
                    bouma_factor=l_bouma.value,
                )

            elif model_type == 'Gaussian':
                layer = gr.GaussianLayer(**grid_kw)
                layer.create_geology(
                    poro_ave=g_poro.value, perm_ave=g_perm.value,
                    poro_std=g_poro_std.value, perm_std=g_perm_std.value,
                    ntg=g_ntg.value,
                    facies_filter=(g_fx.value, g_fy.value, g_fz.value),
                    sand_filter=(g_sx.value, g_sy.value, g_sz.value),
                    nugget=g_nugget.value,
                    poro_perm_corr=g_corr.value,
                )

            elif model_type == 'Meandering Channel':
                layer = gr.MeanderingChannelLayer(**grid_kw)
                layer.create_geology(
                    channel_width=c_width.value,
                    n_channels=c_nch.value,
                    depth_width_ratio=c_dwr.value,
                    friction_coeff=c_friction.value,
                    amplitude=c_ampl.value,
                    slope=c_slope.value,
                    discharge=c_discharge.value,
                    meander_scale=c_meander.value,
                    avulsion_prob=c_avulsion.value,
                    poro_ave=c_poro.value,
                    migration_distance_ratio=c_mig.value,
                    boundary_reflect=c_reflect.value,
                    aggradation_mode=c_aggr.value,
                    nlevel=c_nlevel.value,
                    level_reseed_prob=c_reseed.value,
                    level_jump_ratio=c_jump.value,
                    prob_avul_inside=c_avul_in.value,
                    prob_avul_outside=c_avul_out.value,
                    azimuth=c_azim.value,
                    abandoned_mud_prop=c_abmud.value,
                )

            elif model_type == 'Braided Channel':
                layer = gr.BraidedChannelLayer(**grid_kw)
                layer.create_geology(
                    braidplain_width=b_bpw.value,
                    n_channels=b_nch.value,
                    depth_width_ratio=b_dwr.value,
                    meander_scale=b_meander.value,
                    migration_distance_ratio=b_mig.value,
                    prob_avul_inside=b_avul_in.value,
                    prob_avul_outside=b_avul_out.value,
                    nlevel=(None if b_nlevel.value == 0 else b_nlevel.value),
                    level_reseed_prob=b_reseed.value,
                    level_jump_ratio=b_jump.value,
                    slope=b_slope.value,
                    discharge=b_discharge.value,
                    amplitude=b_ampl.value,
                    friction_coeff=b_friction.value,
                    poro_ave=b_poro.value,
                    azimuth=b_azim.value,
                    abandoned_mud_prop=b_abmud.value,
                )

            elif model_type == 'Delta':
                layer = gr.DeltaLayer(**grid_kw)
                kmin = min(d_kmin.value, d_kmax.value)
                kmax = max(d_kmin.value, d_kmax.value)
                layer.create_geology(
                    feeder_width=d_feeder.value,
                    n_generations=d_ngen.value,
                    fan_angle_deg=d_fan.value,
                    bifurcation_depth=d_bif.value,
                    children_per_split=(kmin, kmax),
                    fan_asymmetry=d_asym.value,
                    azimuth=d_azim.value,
                    apex_x_fraction=d_apex.value,
                    progradation_fraction=d_prog.value,
                    trunk_length_factor=d_trunk.value,
                    length_taper=d_ltap.value,
                    width_taper=d_wtap.value,
                    meander_amp=d_mwig.value,
                    mouth_bar_length_factor=d_mbL.value,
                    mouth_bar_width_factor=d_mbW.value,
                    mouth_bar_hw_ratio=d_mbhw.value,
                    mouth_bar_dw_ratio=d_mbdw.value,
                    dwratio=d_dwr.value,
                    poro_ave=d_poro.value,
                )

            # Select property to plot
            prop = w_prop.value
            if prop == 'Porosity':
                data = layer.poro_mat
                label = f'{model_type} — Porosity'
            elif prop == 'Permeability':
                data = layer.perm_mat
                label = f'{model_type} — Permeability (mD)'
            else:  # Facies
                data = layer.facies.astype(float) if hasattr(layer, 'facies') else layer.active.astype(float)
                label = f'{model_type} — Facies'

            # Choose view type
            view = w_view.value
            mask_z = (prop != 'Facies')
            if view == 'Cube Slices':
                gr.plot_cube_slices(data, title=label, mask_zeros=mask_z)
            elif view == 'Z Slices':
                gr.plot_slices(data, axis=2, title=label, mask_zeros=mask_z)
            elif view == 'X Slices':
                gr.plot_slices(data, axis=0, title=label, mask_zeros=mask_z)
            else:
                gr.plot_slices(data, axis=1, title=label, mask_zeros=mask_z)
            plt.show()

            # Print summary stats
            active_mask = layer.active == 1
            print(f'Grid: ({w_nx.value}, {w_ny.value}, {w_nz.value})')
            print(f'Active fraction: {layer.active.mean():.2f}')
            if active_mask.any():
                print(f'Porosity  — mean: {layer.poro_mat[active_mask].mean():.3f}, std: {layer.poro_mat[active_mask].std():.3f}')
                print(f'Perm (mD) — mean: {layer.perm_mat[active_mask].mean():.2f}, std: {layer.perm_mat[active_mask].std():.2f}')

        except Exception as e:
            print(f'Error: {e}')
            import traceback
            traceback.print_exc()

# ── Connect all widgets to auto-generate ─────────────────────────────
all_widgets = [
    model_selector, w_nx, w_ny, w_nz, w_xlen, w_ylen, w_zlen, w_depth, w_dip,
    w_view, w_prop,
    # Lobe
    l_poro, l_perm, l_poro_std, l_perm_std, l_ntg, l_dhmin, l_dhmax,
    l_rmin, l_rmax, l_asp, l_theta, l_m, l_upthin, l_bouma,
    # Gaussian
    g_poro, g_perm, g_poro_std, g_perm_std, g_ntg,
    g_fx, g_fy, g_fz, g_sx, g_sy, g_sz, g_nugget, g_corr,
    # Meandering Channel
    c_width, c_nch, c_dwr, c_meander, c_avulsion, c_poro,
    c_friction, c_ampl, c_slope, c_discharge, c_mig, c_reflect,
    c_aggr, c_nlevel, c_reseed, c_jump, c_avul_in, c_avul_out, c_azim, c_abmud,
    # Braided Channel
    b_bpw, b_nch, b_dwr, b_meander, b_mig, b_avul_in, b_avul_out,
    b_nlevel, b_reseed, b_jump, b_slope, b_discharge, b_ampl, b_friction, b_poro, b_azim, b_abmud,
    # Delta
    d_feeder, d_ngen, d_fan, d_bif, d_kmin, d_kmax, d_asym, d_azim, d_apex, d_prog,
    d_trunk, d_ltap, d_wtap, d_mwig,
    d_mbL, d_mbW, d_mbhw, d_mbdw, d_dwr, d_poro,
]
for w in all_widgets:
    w.observe(generate_model, names='value')

# ── Layout ────────────────────────────────────────────────────────────
update_params()  # show initial panel

display(widgets.VBox([
    widgets.HTML('<h2>GeoRules Interactive Explorer</h2>'),
    model_selector,
    grid_box,
    param_area,
    view_box,
    output_area
]))